# MVPruner: prune index 2 using saved score index 22
Two GPUs evaluate disjoint test shards, FP32 batch64 per GPU, greedy. Attach cache and a complete base attention export. Score index22 means layer23; pruning occurs after index2 (layer3). Both stress probes must pass before either test shard starts. Metrics are computed once from merged predictions.


In [ ]:
from pathlib import Path
import os, subprocess, sys
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
SOURCE = Path('/kaggle/working/Log_Metrics')
if not SOURCE.exists():
    subprocess.run(['git','clone','--depth','1','--branch','smolvlm-attention',
                    'https://github.com/huyvanzzz/Log_Metrics.git',str(SOURCE)],check=True)
else:
    branch = subprocess.check_output(['git','-C',str(SOURCE),'branch','--show-current'],text=True).strip()
    origin = subprocess.check_output(['git','-C',str(SOURCE),'remote','get-url','origin'],text=True).strip()
    if branch != 'smolvlm-attention' or origin.rstrip('/') != 'https://github.com/huyvanzzz/Log_Metrics.git':
        raise RuntimeError('Source branch/origin mismatch; choose a new source directory')
    if subprocess.check_output(['git','-C',str(SOURCE),'status','--porcelain','--untracked-files=no'],text=True).strip():
        raise RuntimeError('Source has tracked local changes; preserve them before updating')
    subprocess.run(['git','-C',str(SOURCE),'pull','--ff-only','origin','smolvlm-attention'],check=True)
os.chdir(SOURCE)
sys.path.insert(0,str(SOURCE))
subprocess.run([sys.executable,'-m','pip','install','-r',str(SOURCE/'smolvlm_inference/requirements.txt'),
                '-r',str(SOURCE/'smolvlm_inference/requirements-pruning-metrics.txt')],check=True)


In [ ]:
import torch, transformers, tokenizers
assert transformers.__version__ == '4.46.3'
assert tokenizers.__version__ == '0.20.4'
assert torch.cuda.device_count() >= 2, 'Enable two Kaggle GPUs'
from smolvlm_inference.pruning_run import preflight_metrics
preflight_metrics()


In [ ]:
from huggingface_hub import hf_hub_download
import kagglehub
OUTPUT = Path('/kaggle/working/smolvlm_mvpruner_prune2_score22_outputs')
OUTPUT.mkdir(exist_ok=True)
PREPARED = OUTPUT.parent/'mvpruner_prepared'
PREPARED_REVISION = 'd2a6914f681dbb9148c4ac11e6fa92452010506e'
CHECKPOINT_REVISION = 'f9d4e9a4d704078f9e7f39f061d13602cf251c78'
for name in ('manifest.json','train.jsonl','val.jsonl','test.jsonl'):
    hf_hub_download('huyvanzzz/prepared_data',repo_type='dataset',revision=PREPARED_REVISION,
                    filename=name,local_dir=PREPARED,token=False)
CHECKPOINT = hf_hub_download('huyvanzzz/smolvlm256M_base',revision=CHECKPOINT_REVISION,
                            filename='finetune_best.pt',token=False)
CACHE_INPUT_DIR = '' # Optional exact mounted cache directory; nothing is copied.
root = Path(CACHE_INPUT_DIR) if CACHE_INPUT_DIR else Path(kagglehub.dataset_download('huyhuy12334/cache-smolvlm'))
candidates = [p.parent for p in root.rglob('features.float16.memmap')
              if (p.parent/'manifest.json').is_file() and (p.parent/'index.json').is_file()]
if len(candidates) != 1:
    raise RuntimeError(f'Expected one complete cache, found {len(candidates)}')
CACHE = candidates[0].resolve()
RECEIPT = OUTPUT/'validation_receipt.json'

# Folder containing attention.npy, metadata.json, samples.jsonl and manifest.json.
# Attach the existing complete base export; extract ZIP before running.
ATTENTION_INPUT = '/kaggle/input/your-attention-dataset/export'


In [ ]:
from smolvlm_inference.pruning_run import validate_cache, cache_identity
cache = validate_cache(CACHE, PREPARED, cache_identity(), RECEIPT)
env = dict(os.environ)
subprocess.run([sys.executable,'-m','smolvlm_inference.saved_attention_run',
    '--cache',str(CACHE),'--prepared',str(PREPARED),'--receipt',str(RECEIPT),
    '--checkpoint',str(CHECKPOINT),'--attention-input',ATTENTION_INPUT,
    '--config',str(SOURCE/'configs/mvpruner_prune_index_2_score_index_22.json'),
    '--output',str(OUTPUT),'--gpus','2'],check=True,env=env)


In [ ]:
import json
from IPython.display import FileLink, display
print(json.loads((OUTPUT/'metrics_display.json').read_text()))
display(FileLink(str(OUTPUT/'mvpruner_saved_attention_artifacts.zip')))
